# Aspect 2 — Homogeneous vs Heterogeneous GNNs on Relational Data

Does preserving node/edge type information during message passing help, or can a simpler
homogeneous GNN with proper input projections close the gap?

**27 runs** — 3 datasets × 3 architectures × 2 settings (homo/hetero) × 1 seed,
plus 9 parameter-matching ablation runs (homo at hidden_dim=128 ≈ hetero at hidden_dim=64).

## 1. Setup

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

_here = Path(os.path.abspath(""))
ROOT  = _here if _here.name == "aspect2" else _here / "aspect2"
os.chdir(ROOT)

PROCESSED   = ROOT / "processed"
CHECKPOINTS = ROOT / "checkpoints"
RESULTS_CSV = ROOT / "results" / "metrics.csv"
PYTHON      = sys.executable

# ── RESULTS_ONLY: skip preprocessing & training, jump straight to results ──────
RESULTS_ONLY = False

# ── FORCE_RERUN: delete results CSV and re-run all experiments from scratch ────
# Useful when you want to regenerate all results (e.g. after changing training code).
# Has no effect when RESULTS_ONLY=True.
FORCE_RERUN = False

print(f"ROOT: {ROOT}")
print(f"RESULTS_ONLY: {RESULTS_ONLY}   FORCE_RERUN: {FORCE_RERUN}")

ROOT: /home/zeev.kliot/structML_project/aspect2
RESULTS_ONLY: False   FORCE_RERUN: False


## 2. Datasets

Three [RelBench](https://relbench.stanford.edu/) datasets chosen for type diversity:

| Dataset | Task | Node types | Prior expectation |
|---|---|---|---|
| **rel-stack** | user-engagement | 7 | Rich type diversity — hetero should win |
| **rel-avito** | user-visits | 8 | Similar activity streams — homo may close gap |
| **rel-arxiv** | author-category | 6 | Citation graph, multiclass (53 ArXiv categories) |

**Key question:** Across these datasets, does preserving node/edge type information during message passing (hetero) outperform a homogeneous GNN with proper per-type input projections (homo)?

This aligns with the HGB paper (Lv et al., KDD 2021): explicit per-type encoders capture most
of the type-specific information; type-specific message-passing weights add parameters but not necessarily proportional benefit.

rel-arxiv/author-category is a multiclass task (predict author's primary ArXiv category from 53 classes).
Metric: **Macro-F1** instead of AUPRC for this dataset.

In [ ]:
TASKS = [
    ("rel-stack",  "user-engagement"),
    ("rel-avito",  "user-visits"),
    ("rel-arxiv",  "author-category"),
]

for ds, task in TASKS:
    meta_path = PROCESSED / ds / task / "meta.json"
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        n_types = len(meta["node_types"])
        n_edges = len([e for e in meta["edge_types"] if not e[1].startswith("rev_")])
        task_type = meta.get("task_type", "binary")
        num_classes = meta.get("num_classes", 1)
        print(f"{ds}/{task}: {n_types} node types, {n_edges} FK edges  [{task_type}, {num_classes} class(es)]")
    else:
        print(f"{ds}/{task}: not yet preprocessed")

## 3. Preprocessing

Builds `HeteroData` objects (train/val/test splits) with temporal masking and standardized features.
Output: `processed/{dataset}/{task}/{split}.pt` + `meta.json`.
Skips datasets already preprocessed (checks for `meta.json`).

In [3]:
if RESULTS_ONLY:
    print("RESULTS_ONLY=True — skipping preprocessing.")
else:
    all_done = all(
        (PROCESSED / ds / task / "meta.json").exists()
        for ds, task in TASKS
    )
    if all_done:
        print("All datasets already preprocessed — skipping.")
    else:
        print("Running preprocess.py …")
        subprocess.run([PYTHON, "preprocess.py"], cwd=ROOT, check=True)

All datasets already preprocessed — skipping.


## 4. Model Architectures

**Homo** — *Homogeneous baseline* (HGB KDD 2021 standard):
1. Per-type `nn.Linear(feat_dim → hidden_dim)` projections — all node types get the same dimensionality
2. Concatenate all nodes into a single tensor; remap edge indices globally
3. Shared-weight GNN (SAGEConv / GATConv / HGTConv with single node+edge type)

**Hetero** — *Type-aware GNN*:
- SAGE/GAT: `to_hetero(conv, metadata)` per layer — separate weight matrix per (src, edge, dst) triple
- HGT: per-type input encoder + `HGTConv` natively uses type-specific attention

**Parameter-matching ablation**: hetero at h=64 vs homo at h=128.  
Asks: *is hetero better because of type awareness, or just more parameters?*

## 5. Experiment Configuration

In [ ]:
ARCHS      = ["sage", "gat", "hgt"]
SETTINGS   = ["homo", "hetero", "homo_noenc", "hybrid"]
LAYERS = [2, 3]
SEED       = 0
HIDDEN_DIM_MAIN     = 64
HIDDEN_DIM_ABLATION = 128

def ckpt_path(dataset, task, arch, setting, num_layers=2, hidden_dim=HIDDEN_DIM_MAIN):
    return CHECKPOINTS / dataset / task / f"{arch}_{setting}_h{hidden_dim}_L{num_layers}_s{SEED}.pt"

# Count checkpoint status
all_configs = [
    (ds, task, arch, setting, L, HIDDEN_DIM_MAIN)
    for ds, task in TASKS
    for arch in ARCHS
    for setting in SETTINGS
    for L in LAYERS
] + [
    (ds, task, arch, "homo", L, HIDDEN_DIM_ABLATION)
    for ds, task in TASKS
    for arch in ARCHS
    for L in LAYERS
]

done    = [(ds, task, arch, s, L, h) for ds, task, arch, s, L, h in all_configs if ckpt_path(ds, task, arch, s, L, h).exists()]
pending = [(ds, task, arch, s, L, h) for ds, task, arch, s, L, h in all_configs if not ckpt_path(ds, task, arch, s, L, h).exists()]

print(f"Total configs: {len(all_configs)}  |  done: {len(done)}  |  pending: {len(pending)}")
if pending:
    print("\nPending:")
    for ds, task, arch, s, L, h in pending:
        print(f"  {ds}/{task}  {arch}  {s}  L={L}  h={h}")

Total configs: 90  |  done: 90  |  pending: 0


## 6. Training

The cell below runs all pending configurations. Completed ones are automatically skipped unless `FORCE_RERUN=True`.

In [5]:
if RESULTS_ONLY:
    print("RESULTS_ONLY=True — skipping training.")
else:
    if FORCE_RERUN and RESULTS_CSV.exists():
        import shutil
        backup = RESULTS_CSV.with_suffix(".csv.bak")
        shutil.copy(RESULTS_CSV, backup)
        RESULTS_CSV.unlink()
        print(f"FORCE_RERUN=True — backed up and deleted {RESULTS_CSV.name}, re-running all configs.")

    cmd = [PYTHON, "-u", "run_experiments.py"]
    if FORCE_RERUN:
        cmd.append("--force")
    elif not pending:
        print("All runs complete — skipping training.")
        cmd = None

    if cmd:
        subprocess.run(cmd, cwd=ROOT, check=True)

All runs complete — skipping training.


## 7. Results

In [ ]:
# Load and deduplicate (keep last run per config in case of restarts)
df = pd.read_csv(RESULTS_CSV)
df_all = df.drop_duplicates(
    subset=["dataset", "task", "setting", "arch", "num_layers", "hidden_dim", "seed"],
    keep="last"
).copy()

SETTING_LABELS = {"homo": "Homo", "hetero": "Hetero", "homo_noenc": "Homo (no enc)", "hybrid": "Hybrid"}
ARCH_LABELS    = {"sage": "SAGE", "gat": "GAT", "hgt": "HGT"}
DATASET_LABELS = {
    "rel-stack":  "rel-stack\n(user-engagement)",
    "rel-avito":  "rel-avito\n(user-visits)",
    "rel-arxiv":  "rel-arxiv\n(author-category)",
}

df_all["Setting"] = df_all["setting"].map(SETTING_LABELS)
df_all["Arch"]    = df_all["arch"].map(ARCH_LABELS)

# Choose primary metric: AUPRC for binary, macro_f1 for multiclass
def primary_metric(row):
    if pd.notna(row.get("test_macro_f1")) and row.get("test_macro_f1", 0) > 0:
        return row["test_macro_f1"]
    return row["test_auprc"]

df_all["test_metric"] = df_all.apply(primary_metric, axis=1)

# Main runs only (h=64)
df_main = df_all[df_all["hidden_dim"] == 64].copy()
# Ablation (homo h=128 vs hetero h=64)
df_ablation = df_all[
    ((df_all["setting"] == "homo") & (df_all["hidden_dim"] == 128)) |
    ((df_all["setting"] == "hetero") & (df_all["hidden_dim"] == 64))
].copy()
df_ablation["label"] = df_ablation.apply(
    lambda r: f"{r['Arch']}-{r['Setting']}(h={r['hidden_dim']})", axis=1
)

print(f"Main runs loaded: {len(df_main)}")
print(f"Ablation runs loaded: {len(df_ablation)}")
df_main[["dataset", "arch", "setting", "hidden_dim", "test_metric", "num_params"]].sort_values(
    ["dataset", "arch", "setting"]
)

In [ ]:
# Figure 1: Primary metric by setting × arch, faceted by dataset
datasets   = [ds for ds, _ in TASKS]
archs      = ["SAGE", "GAT", "HGT"]
settings   = ["homo", "hetero", "homo_noenc", "hybrid"]
colors     = {"homo": "#4C72B0", "hetero": "#DD8452",
              "homo_noenc": "#55A868", "hybrid": "#C44E52"}
x          = np.arange(len(archs))
width      = 0.2

fig, axes = plt.subplots(1, len(datasets), figsize=(6 * len(datasets), 4), sharey=False)
fig.suptitle("Test Metric by Setting and Architecture (best across depths)", fontsize=13, y=1.02)

for ax, ds in zip(axes, datasets):
    sub = df_main[df_main["dataset"] == ds]
    metric_name = "Macro-F1" if ds == "rel-arxiv" else "AUPRC"
    for i, setting in enumerate(settings):
        vals = []
        for arch in ["sage", "gat", "hgt"]:
            rows = sub[(sub["setting"] == setting) & (sub["arch"] == arch)]
            vals.append(float(rows["test_metric"].max()) if len(rows) else 0.0)
        offset = (i - (len(settings) - 1) / 2) * width
        bars = ax.bar(x + offset, vals, width, label=SETTING_LABELS[setting], color=colors[setting])
        for bar, v in zip(bars, vals):
            if v > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                        f"{v:.3f}", ha="center", va="bottom", fontsize=7)
    ax.set_title(DATASET_LABELS[ds], fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(archs)
    ax.set_ylabel(f"Test {metric_name}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(ROOT / "results" / "fig1_metric_by_setting.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 2: Parameter efficiency — test metric vs param count
# Each point = one run (arch × setting × hidden_dim). Color = setting, marker = arch.
setting_colors = {"homo": "#4C72B0", "hetero": "#DD8452",
                  "homo_noenc": "#55A868", "hybrid": "#C44E52"}
arch_markers   = {"sage": "o", "gat": "^", "hgt": "s"}

fig, axes = plt.subplots(1, len(datasets), figsize=(7 * len(datasets), 6), sharey=False)
fig.suptitle("Figure 2: Parameter Efficiency (test metric vs param count)", fontsize=14, y=1.02)

for ax, ds in zip(axes, datasets):
    sub = df_all[df_all["dataset"] == ds]
    metric_name = "Macro-F1" if ds == "rel-arxiv" else "AUPRC"
    for _, row in sub.iterrows():
        ax.scatter(row["num_params"] / 1_000, row["test_metric"],
                   color=setting_colors[row["setting"]],
                   marker=arch_markers[row["arch"]],
                   s=90, alpha=0.85, zorder=3)
        ax.annotate(f"h={int(row['hidden_dim'])}",
                    (row["num_params"] / 1_000, row["test_metric"]),
                    textcoords="offset points", xytext=(5, 2), fontsize=7, alpha=0.8)
    ax.set_xlabel("Parameter Count (k)")
    ax.set_ylabel(f"Test {metric_name}")
    ax.set_title(DATASET_LABELS[ds], fontsize=11, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

handles = [
    plt.scatter([], [], color=setting_colors[s], marker=arch_markers[a],
                s=70, label=f"{SETTING_LABELS[s]} / {ARCH_LABELS[a]}")
    for s in ["homo", "hetero", "homo_noenc", "hybrid"]
    for a in ["sage", "gat", "hgt"]
]
axes[-1].legend(handles=handles, fontsize=8, title="Setting / Arch",
                loc="center left", bbox_to_anchor=(1.02, 0.5), framealpha=0.8)

plt.tight_layout()
plt.savefig(ROOT / "results" / "fig2_param_efficiency.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Figure 3: Parameter-matched comparison — homo h=128 vs hetero h=64
if len(df_ablation) > 0:
    fig, axes = plt.subplots(1, len(datasets), figsize=(5 * len(datasets), 4), sharey=False)
    fig.suptitle("Parameter-Matched: Homo(h=128) vs Hetero(h=64)", fontsize=13, y=1.02)

    abl_colors = {"homo": "#4C72B0", "hetero": "#DD8452"}
    for ax, ds in zip(axes, datasets):
        sub = df_ablation[df_ablation["dataset"] == ds]
        metric_name = "Macro-F1" if ds == "rel-arxiv" else "AUPRC"
        archs_present = sorted(sub["arch"].unique())
        x_abl = np.arange(len(archs_present))
        for i, (setting, hdim) in enumerate([("homo", 128), ("hetero", 64)]):
            vals = []
            for arch in archs_present:
                row = sub[(sub["setting"] == setting) & (sub["arch"] == arch) & (sub["hidden_dim"] == hdim)]
                vals.append(float(row["test_metric"].values[0]) if len(row) else 0.0)
            lbl = "Homo h=128" if setting == "homo" else "Hetero h=64"
            bars = ax.bar(x_abl + (i - 0.5) * 0.35, vals, 0.35,
                          label=lbl, color=abl_colors[setting])
        ax.set_title(DATASET_LABELS[ds], fontsize=10)
        ax.set_xticks(x_abl)
        ax.set_xticklabels([ARCH_LABELS.get(a, a) for a in archs_present])
        ax.set_ylabel(f"Test {metric_name}")
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(ROOT / "results" / "fig3_param_matched.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Ablation runs not yet complete.")

## 8. Analysis & Conclusions

## 9. Effective Parameters

Not all parameters in a k-layer GNN actually influence the output. Only nodes and edge types
within **k hops** of the target can propagate information to the prediction.

- **Homo**: encoder for node type X is *wasted* if X is more than `num_layers` hops from target
- **Hetero**: weight matrix for edge type (A, e, B) at GNN layer l is *wasted* if B is more than `num_layers − l` hops from target

The hetero model allocates separate weights per (src, edge, dst) triple — with many edge types and
a shallow GNN, a large fraction of those weights sit on paths that can never reach the target.

In [ ]:
# Run effective parameter analysis (generates figures to results/figures/)
# effective_params.py computes hop-reachability per schema and classifies each
# parameter as effective or wasted for both homo and hetero, at L=2 and L=3.
import subprocess
result = subprocess.run(
    [PYTHON, "effective_params.py"],
    cwd=ROOT, capture_output=False
)
if result.returncode != 0:
    print("effective_params.py failed — check output above.")

In [ ]:
from IPython.display import Image, display

fig_dir = ROOT / "results" / "figures"

print("Figure: Test AUPRC vs Effective Parameter Count")
display(Image(str(fig_dir / "auprc_vs_effective_params.png")))

print("\nFigure: Effective vs Wasted Parameters (L=2, stacked by component)")
display(Image(str(fig_dir / "effective_params_stacked.png")))

print("\nFigure: Schema Reachability — Node hop distance from target")
display(Image(str(fig_dir / "schema_reachability.png")))

print("\nFigure: L=2 vs L=3 — How depth changes the wasted parameter fraction")
display(Image(str(fig_dir / "effective_params_depth_comparison.png")))

In [ ]:
print("=" * 70)
print("SUMMARY TABLE — Primary Metric (main grid, h=64)")
print("(binary tasks: AUPRC; multiclass tasks: Macro-F1)")
print("=" * 70)
summary = df_main.pivot_table(
    values="test_metric",
    index=["dataset", "arch"],
    columns="setting",
    aggfunc="mean"
).round(4)
summary["hetero_gain"] = (summary["hetero"] - summary["homo"]).round(4)
print(summary.to_string())

print("\n" + "=" * 70)
print("Parameter counts (rel-stack, h=64)")
print("=" * 70)
param_summary = df_main[df_main["dataset"] == "rel-stack"][
    ["arch", "setting", "num_params"]
].sort_values(["arch", "setting"])
print(param_summary.to_string(index=False))